# Executive Summary – Local Test Notebook

**Standalone** – identical logic to `scripts/executive_summary.py`.
Does not import or reference it; both files are maintained independently.

**Environment detection**
- `is_running_in_airflow()` = `True` only when Airflow is installed AND
  `AIRFLOW_CTX_DAG_ID` env var is set. In a notebook this is always `False`.
- Email is **not sent** locally; Cell 8 saves an HTML preview file instead.

**Steps**
1. Cell 1 – install local deps (once)
2. Cell 2 – set credentials
3. Cell 3 – all shared helpers (GP, OpenAI, SQL, prompt, email builders)
4. Cell 4 – choose month
5. Cell 5 – query GP and inspect data
6. Cell 6 – generate summary
7. Cell 7 – save to .txt (optional)
8. Cell 8 – preview email HTML locally (no send)

In [ ]:
# ── Cell 1: Install local dependencies (run once if needed) ────────────────────
# !pip install psycopg2-binary openai pandas

In [ ]:
# ── Cell 2: Credentials ─────────────────────────────────────────────────
#
# Set env vars BEFORE Cell 3. Ignored when is_running_in_airflow() is True.

import os
import getpass

os.environ['GP_HOST']     = 'greenplum-rdsp.zur.swissbank.com'
os.environ['GP_PORT']     = '5432'
os.environ['GP_DB']       = 'gprdsp'
os.environ['GP_USER']     = 'ds_rdsp_dev'
os.environ['GP_SCHEMA']   = 'core_ikg'
os.environ['GP_PASSWORD'] = getpass.getpass('Greenplum password: ')

os.environ['OPENAI_API_KEY']  = getpass.getpass('OpenAI / Azure API key: ')
os.environ['OPENAI_BASE_URL'] = 'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/'

print('Credentials stored in environment variables.')

In [ ]:
# ── Cell 3: All shared helpers ──────────────────────────────────────────────
# Identical logic to scripts/executive_summary.py
# (connection, SQL, prompt, LLM, email builders)

from __future__ import annotations

import os, re, logging
from datetime import datetime
from typing import Optional

import pandas as pd
from openai import OpenAI


# ── config.py constants (mirrors scripts/config.py) ───────────────────────────
GREENPLUM_HOST = 'greenplum-rdsp.zur.swissbank.com'
GREENPLUM_PORT = 5432
GREENPLUM_DB   = 'gprdsp'
GREENPLUM_USER = 'ds_rdsp_dev'


# ── Environment detection ────────────────────────────────────────────────
try:
    from airflow.models import Connection, Variable                      # type: ignore
    from airflow.configuration import conf                               # type: ignore
    _HAS_AIRFLOW = True
except Exception:
    Variable = Connection = conf = None  # type: ignore
    _HAS_AIRFLOW = False


def is_running_in_airflow() -> bool:
    return _HAS_AIRFLOW and bool(os.environ.get('AIRFLOW_CTX_DAG_ID'))


print(f'Airflow available      : {_HAS_AIRFLOW}')
print(f'Running inside Airflow : {is_running_in_airflow()}')
print('Mode:', 'PRODUCTION (Airflow)' if is_running_in_airflow() else 'LOCAL (env vars)')


# ── Model & email constants ───────────────────────────────────────────────
MODEL_NAME  = 'gpt-4.1'
MAX_TOKENS  = 12000
TEMPERATURE = 0.1

POSTGRES_CONN_ID_VAR    = 'GP_Dash_connect'
IKG_SCHEMA_VAR          = 'IKG_DASHBOARD_SCHEMA'
OPENAI_CONN_ID          = 'STAAT-DS-OPENAI-LLM'
DEFAULT_OPENAI_BASE_URL = 'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/'
DEFAULT_GP_SCHEMA       = 'core_ikg'

EMAIL_RECIPIENTS_VAR   = 'staat_monthly_executive_summary_email'
LOCAL_EMAIL_RECIPIENTS = ['pratik.shah.2@ubs.com']
EMAIL_FROM_ADDR        = 'staat-insights@ubs.com'

_client: Optional[OpenAI] = None

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ── Connection helpers ─────────────────────────────────────────────────
def _get_openai_client() -> OpenAI:
    global _client
    if _client is not None: return _client
    if is_running_in_airflow():
        conn = Connection.get_connection_from_secrets(OPENAI_CONN_ID)
        api_key, base_url = conn.password, conn.host
    else:
        api_key  = os.environ['OPENAI_API_KEY']
        base_url = os.environ.get('OPENAI_BASE_URL', DEFAULT_OPENAI_BASE_URL)
    _client = OpenAI(api_key=api_key, base_url=base_url)
    return _client


def _get_gp_conn(allow_prompt: bool = True):
    if is_running_in_airflow():
        from airflow.providers.postgres.hooks.postgres import PostgresHook  # type: ignore
        return PostgresHook(postgres_conn_id=Variable.get(POSTGRES_CONN_ID_VAR)).get_conn()
    import psycopg2
    password = os.environ.get('GP_PASSWORD')
    if not password:
        password = getpass.getpass('Enter Greenplum password: ') if allow_prompt \
                   else (_ for _ in ()).throw(RuntimeError('GP_PASSWORD required.'))
    return psycopg2.connect(
        host=os.environ.get('GP_HOST', GREENPLUM_HOST),
        port=int(os.environ.get('GP_PORT', GREENPLUM_PORT)),
        dbname=os.environ.get('GP_DB', GREENPLUM_DB),
        user=os.environ.get('GP_USER', GREENPLUM_USER),
        password=password,
    )


def _get_schema() -> str:
    return Variable.get(IKG_SCHEMA_VAR) if is_running_in_airflow() \
           else os.environ.get('GP_SCHEMA', DEFAULT_GP_SCHEMA)


# ── Environment & email helpers ──────────────────────────────────────────
def _get_environment() -> str:
    """Return 'Dev', 'UAT', or 'Prod' (matches existing DAG convention)."""
    if not is_running_in_airflow(): return 'Dev'
    try:
        base_url = conf.get('webserver', 'base_url')
        if base_url == Variable.get('AIRFLOW_UAT_URL',  default_var=''): return 'Dev'
        if base_url == Variable.get('AIRFLOW_PROD_URL', default_var=''): return 'Prod'
    except Exception as e:
        logger.warning('Could not determine environment: %s', e)
    return 'Dev'


def get_email_recipients() -> list:
    """Dev/local Airflow → LOCAL_EMAIL_RECIPIENTS.  Prod → Airflow Variable."""
    if not is_running_in_airflow(): return []
    if _get_environment() != 'Prod': return LOCAL_EMAIL_RECIPIENTS
    try:
        raw = Variable.get(EMAIL_RECIPIENTS_VAR, default_var='')
        r   = [x.strip() for x in raw.split(',') if x.strip()]
        return r if r else LOCAL_EMAIL_RECIPIENTS
    except Exception as e:
        logger.warning('Could not read %s: %s', EMAIL_RECIPIENTS_VAR, e)
        return LOCAL_EMAIL_RECIPIENTS


# ── SQL helpers ──────────────────────────────────────────────────────────
def _build_month_query(month_value: str, schema: str) -> str:
    staat = f'{schema}.staat_insight_release'
    odm   = f'{schema}.odm_release_details'
    return f"""
    WITH target_iterations AS (
        SELECT iteration_end_date, MAX(batch) AS max_batch
        FROM {staat}
        WHERE TO_CHAR(prod_release_date::date, 'YYYY-MM') = '{month_value}'
        GROUP BY iteration_end_date
    ),
    staat_latest AS (
        SELECT s.* FROM {staat} s
        INNER JOIN target_iterations ti
            ON s.iteration_end_date = ti.iteration_end_date AND s.batch = ti.max_batch
    ),
    odm_latest AS (
        SELECT o.* FROM {odm} o
        INNER JOIN target_iterations ti
            ON o.iteration_end_date = ti.iteration_end_date AND o.batch = ti.max_batch
    )
    SELECT
        s.id_x, s.title, s.labels, s.issue_summary, s.state, s.weight,
        s.prod_release_date, s.iteration_end_date, s.iteration_start_date, s.batch,
        o.rule_name, o.target_type, o.change_type
    FROM staat_latest s
    LEFT JOIN odm_latest o
        ON s.id_x = o.issue_id AND s.iteration_end_date = o.iteration_end_date
    ORDER BY s.iteration_end_date, s.id_x
    """


def _build_reactivated_query(insight_types: list, schema: str) -> str:
    staat     = f'{schema}.staat_insight_release'
    odm       = f'{schema}.odm_release_details'
    in_clause = ', '.join(f"'{t}'" for t in insight_types)
    return f"""
    WITH odm_ranked AS (
        SELECT o.*,
               MAX(o.batch) OVER (PARTITION BY o.iteration_end_date) AS max_batch
        FROM {odm} o WHERE o.rule_name IN ({in_clause})
    ),
    odm_latest AS (SELECT * FROM odm_ranked WHERE batch = max_batch)
    SELECT
        o.issue_id AS id_x, o.iteration_end_date, o.batch,
        o.rule_name, o.target_type, o.change_type,
        s.title, s.issue_summary, s.labels, s.state, s.weight,
        s.prod_release_date, s.iteration_start_date
    FROM odm_latest o
    LEFT JOIN {staat} s
        ON o.issue_id = s.id_x AND o.iteration_end_date = s.iteration_end_date
           AND o.batch = s.batch
    ORDER BY o.rule_name, o.iteration_end_date, o.issue_id
    """


# ── Data helpers ───────────────────────────────────────────────────────────
def get_reactivated_insights(schema: str, conn, year: int, month: int) -> pd.DataFrame:
    sql = f"""
        SELECT DISTINCT insight_type
        FROM {schema}.odm_exclusion_insight_type
        WHERE EXTRACT(YEAR FROM last_upd_dte::date) = {year}
          AND EXTRACT(MONTH FROM last_upd_dte::date) = {month}
          AND is_curr = 0
    """
    try:
        df = pd.read_sql_query(sql, conn)
    except Exception as e:
        logger.warning('[reactivated] exclusion query failed: %s', e); return pd.DataFrame()
    if df.empty: return pd.DataFrame()
    types = df['insight_type'].dropna().str.strip().tolist()
    logger.info('[reactivated] Found %d type(s): %s', len(types), types)
    try:
        d = pd.read_sql_query(_build_reactivated_query(types, schema), conn)
        d['change_type'] = d['change_type'].replace({'added': 'new'})
        return d
    except Exception as e:
        logger.warning('[reactivated] detail query failed: %s', e); return pd.DataFrame()


def get_exec_summary_data(month_value: str, allow_prompt: bool = True) -> tuple:
    schema = _get_schema(); conn = _get_gp_conn(allow_prompt)
    year, mon = (int(x) for x in month_value.split('-'))
    cur = pd.read_sql_query(_build_month_query(month_value, schema), conn)
    cur['change_type'] = cur['change_type'].replace({'added': 'new'})
    react = get_reactivated_insights(schema, conn, year, mon)
    try:
        nxt = pd.read_sql_query(_build_month_query(str(pd.Period(month_value,'M')+1), schema), conn)
        nxt['change_type'] = nxt['change_type'].replace({'added': 'new'})
        next_df = nxt if not nxt.empty else None
    except Exception: next_df = None
    conn.close()
    return cur, next_df, react


def get_month_options_from_db() -> list:
    schema = _get_schema(); conn = _get_gp_conn()
    df = pd.read_sql_query(
        f"""SELECT DISTINCT TO_CHAR(prod_release_date::date,'YYYY-MM') AS month_val
            FROM {schema}.staat_insight_release WHERE prod_release_date IS NOT NULL
            ORDER BY month_val DESC""", conn)
    conn.close()
    opts = []
    for v in df['month_val']:
        try: opts.append({'label': pd.Period(v,'M').to_timestamp().strftime('%B %Y'), 'value': v})
        except Exception: pass
    return opts


# ── Label helpers ──────────────────────────────────────────────────────────
def _has_label(v, t):
    return isinstance(v,str) and t.lower() in [x.strip().lower() for x in v.split(',')]
def _filter_by_label(df, label):
    if df is None or df.empty or 'labels' not in df: return pd.DataFrame()
    return df.loc[df['labels'].apply(lambda v: _has_label(v, label))]


# ── Prompt helpers ───────────────────────────────────────────────────────
def _build_reactivated_text(df, skip_rules=None):
    if df is None or df.empty: return 'No reactivated insights found.'
    skip = skip_rules or set(); lines = []; rn = 0
    for rule, grp in df.groupby('rule_name', sort=True):
        if rule in skip: continue
        rn += 1; lines.append(f'Reactivated Insight #{rn}: {rule}')
        seen: set = set(); sn = 0
        for _, row in grp.iterrows():
            idx = row.get('id_x','')
            if idx and idx not in seen:
                seen.add(idx); sn += 1
                lines += [f'  Story #{sn}:', f'    Title:   {row.get("title","N/A")}',
                           f'    Summary: {row.get("issue_summary","N/A")}']
        lines.append('')
    return '\n'.join(lines).strip() or 'No reactivated insights found.'


def _build_prompt(issues_df, next_month_df=None, reactivated_df=None):
    issues_lines=[]; seen_ids=set()
    for _,row in issues_df.iterrows():
        i=row.get('id_x','')
        if i and i not in seen_ids:
            seen_ids.add(i)
            issues_lines.append(f"Story #{len(issues_lines)+1}:\nTitle:   {row.get('title','N/A')}\n"
                                 f"Summary: {row.get('issue_summary','N/A')}\nState:   {row.get('state','N/A')}\n"
                                 f"Labels:  {row.get('labels','N/A')}\nWeight:  {row.get('weight','N/A')}")
    issues_text='\n\n'.join(issues_lines[:20]) or 'No issues data available.'
    top_df=_filter_by_label(issues_df,'Top Feature'); top_lines=[]; seen_top=set()
    for _,row in top_df.iterrows():
        i=row.get('id_x','')
        if i and i not in seen_top:
            seen_top.add(i)
            top_lines.append(f"Top Feature #{len(top_lines)+1}:\nTitle:{row.get('title','N/A')}\n"
                              f"Summary:{row.get('issue_summary','N/A')}\nRule:{row.get('rule_name','N/A')}")
    top_text='\n\n'.join(top_lines[:5]) or "No 'Top Feature' found."
    ni_df=_filter_by_label(issues_df,'New Insight'); ni_lines=[]; seen_rules=set()
    for _,row in ni_df.iterrows():
        r=row.get('rule_name','')
        if r and r not in seen_rules:
            seen_rules.add(r)
            ni_lines.append(f"New Insight #{len(ni_lines)+1}:\nRule Name: {r}\n"
                             f"Title: {row.get('title','N/A')}\nSummary: {row.get('issue_summary','N/A')}")
    ni_text='\n\n'.join(ni_lines[:15]) or "No 'New Insight' labelled issues found."
    react_text=_build_reactivated_text(reactivated_df, skip_rules=seen_rules)
    next_lines=[]
    if next_month_df is not None and not next_month_df.empty:
        ns=set()
        for _,row in _filter_by_label(next_month_df,'New Insight').iterrows():
            k=row.get('rule_name','') or row.get('title','')
            if k and k not in ns: ns.add(k); next_lines.append(f"- {row.get('title',k)}")
    next_text='\n'.join(next_lines[:10]) or 'No next-month data – use current trends.'
    return f"""
    You are an expert product manager. Generate a comprehensive executive summary report.

    === GITLAB ISSUES DATA ===\n{issues_text}
    === TOP FEATURE DATA ===\n{top_text}
    === NEW INSIGHTS DATA ===\n{ni_text}
    === REACTIVATED INSIGHTS ===\nInclude in Section 3 as Reactivated.\n{react_text}

    === REQUIRED OUTPUT FORMAT ===
    1. Executive Summary:
    - What materially changed this month: <answer>
    - Why it matters to the business: <answer>

    2. Top Feature (or Insight) of the Month:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>

    3. New Insights (ranked by importance/impact, including reactivated insights):
    For each new or reactivated insight:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>
    - Status: New | Reactivated

    4. Process Improvements & Optimization:
    - Name / brief description: <answer>
    - How did we do it?: <answer>
    - Benefits: <answer>

    5. Platform Maintenance & Stability:
    - Maintenance, fixes, or technical improvements: <answer>
    - Why this matters: <answer>

    6. New Insights \u2013 Coming Soon:
    {next_text}
    - Focus areas (2-4 items max): list Insight Title only

    === INSTRUCTIONS ===
    - Be concise and business-focused. Rank by impact.
    - Section 1: 2 sentences per question.
    - Section 3: include reactivated insights at end, marked Status: Reactivated.
    - Section 6: Insight Titles only; if no data, give directional focus areas.
    - Do NOT include instruction notes in output.
    """.strip()


# ── LLM call ─────────────────────────────────────────────────────────────
def generate_summary(issues_df, next_month_df=None, reactivated_df=None) -> str:
    r = _get_openai_client().chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role':'user','content':_build_prompt(issues_df,next_month_df,reactivated_df)}],
        max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    return r.choices[0].message.content.strip()


def build_save_content(text: str, month_label: str) -> str:
    return (f'Executive Summary Report \u2013 {month_label}\n'
            f'Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
            + '='*80 + '\n\n' + text)


# ── Email HTML builder ──────────────────────────────────────────────────
_SECTION_COLORS = [
    '#002060', '#0072CE', '#006341', '#C55A11', '#7B2D8B', '#156082',
]

def _format_summary_as_html(summary_text: str) -> str:
    lines=summary_text.strip().split('\n'); parts=[]
    in_ul=False; in_sec=False; cur_col=_SECTION_COLORS[0]

    def close_ul():
        nonlocal in_ul
        if in_ul: parts.append('</ul>'); in_ul=False

    def close_sec():
        nonlocal in_sec
        if in_sec: close_ul(); parts.extend(['  </div>','</div>']); in_sec=False

    def open_ul():
        nonlocal in_ul
        if not in_ul:
            parts.append('<ul style="margin:8px 0 4px 0;padding:0;list-style:none;">')
            in_ul=True

    def status_badge(val):
        v=val.lower()
        if 'reactivated' in v: bg,fg='#fff3e0','#c55a11'
        elif 'new' in v:        bg,fg='#e8f5e9','#2e7d32'
        else:                   bg,fg='#e3f2fd','#0072ce'
        return (f'<span style="background:{bg};color:{fg};padding:2px 10px;'
                f'border-radius:12px;font-size:11px;font-weight:700;'
                f'border:1px solid {fg}40;">{val}</span>')

    for line in lines:
        s=line.rstrip()
        if not s: close_ul(); continue
        m=re.match(r'^(\d+)\.\s+(.+)',s)
        if m:
            close_sec()
            num=int(m.group(1)); title=m.group(2).rstrip(':').strip()
            col=_SECTION_COLORS[(num-1)%len(_SECTION_COLORS)]; cur_col=col
            parts.append(f'<div style="margin-bottom:20px;border-left:4px solid {col};'
                         f'background:#fff;border-radius:0 8px 8px 0;'
                         f'box-shadow:0 1px 4px rgba(0,0,0,0.06);">')
            parts.append(f'  <div style="background:linear-gradient(to right,{col}14,transparent);'
                         f'padding:11px 20px;border-bottom:1px solid {col}22;">')
            parts.append(f'    <h2 style="margin:0;font-size:15px;font-weight:700;color:{col};line-height:1.3;">'
                         f'<span style="background:{col};color:#fff;border-radius:50%;'
                         f'display:inline-flex;align-items:center;justify-content:center;'
                         f'width:24px;height:24px;font-size:12px;font-weight:800;'
                         f'margin-right:10px;flex-shrink:0;">{num}</span>{title}</h2>')
            parts.extend(['  </div>','  <div style="padding:12px 20px 8px 20px;">'])
            in_sec=True; continue
        bm=re.match(r'^[-\u2022\u25b8]\s+(.+)',s)
        if bm:
            content=bm.group(1).strip(); open_ul()
            kv=re.match(r'^([^:]+?):\s*(.+)',content)
            if kv:
                key=kv.group(1).strip(); val=kv.group(2).strip()
                val_html=status_badge(val) if key.lower()=='status' \
                         else f'<span style="color:#444;line-height:1.5;">{val}</span>'
                parts.append(f'<li style="padding:6px 0;border-bottom:1px solid #f4f4f4;'
                             f'font-size:13.5px;display:flex;align-items:flex-start;gap:4px;">'
                             f'<strong style="color:#222;min-width:190px;flex-shrink:0;'
                             f'padding-right:8px;">{key}:</strong>{val_html}</li>')
            else:
                parts.append(f'<li style="padding:6px 0;font-size:13.5px;color:#333;'
                             f'line-height:1.5;display:flex;align-items:flex-start;">'
                             f'<span style="color:{cur_col};margin-right:8px;font-size:11px;'
                             f'padding-top:3px;">&#9658;</span><span>{content}</span></li>')
            continue
        close_ul()
        parts.append(f'<p style="margin:6px 0 4px 0;font-size:13px;'
                     f'color:#666;font-style:italic;">{s}</p>')
    close_sec()
    return '\n'.join(parts)


def build_email_html(summary_text: str, month_label: str, environment: str) -> str:
    badge={'Prod':('#c41e3a','#fff'),'UAT':('#e07b00','#fff'),'Dev':('#28a745','#fff')}
    bg,fg=badge.get(environment,('#003087','#fff'))
    gen=datetime.now().strftime('%B %d, %Y at %H:%M')
    body=_format_summary_as_html(summary_text)
    return f"""<!DOCTYPE html>
<html lang=\"en\"><head><meta charset=\"UTF-8\">
<meta name=\"viewport\" content=\"width=device-width,initial-scale=1.0\">
<title>Executive Summary \u2013 {month_label}</title></head>
<body style=\"margin:0;padding:0;background:#eef1f6;
     font-family:'Helvetica Neue',Helvetica,Arial,sans-serif;\">
<table width=\"100%\" cellpadding=\"0\" cellspacing=\"0\" style=\"background:#eef1f6;\">
  <tr><td align=\"center\" style=\"padding:28px 12px;\">
    <table width=\"660\" cellpadding=\"0\" cellspacing=\"0\"
           style=\"max-width:660px;background:#fff;border-radius:10px;
                  overflow:hidden;box-shadow:0 4px 18px rgba(0,0,0,0.12);\">
      <tr><td style=\"background:linear-gradient(135deg,#001a4d 0%,#002e6e 55%,#00529b 100%);
                     padding:30px 36px 26px 36px;\">
        <table width=\"100%\" cellpadding=\"0\" cellspacing=\"0\"><tr>
          <td><div style=\"font-size:10.5px;font-weight:600;color:#7eb3e0;
                          letter-spacing:2px;text-transform:uppercase;margin-bottom:8px;\">STAAT Insights | Monthly Report</div>
              <h1 style=\"margin:0 0 6px;font-size:26px;font-weight:800;
                          color:#fff;letter-spacing:-0.5px;line-height:1.2;\">Executive Summary</h1>
              <div style=\"font-size:16px;color:#a8cef0;font-weight:400;margin-top:4px;\">{month_label}</div></td>
          <td style=\"text-align:right;vertical-align:top;padding-left:16px;\">
            <span style=\"background:{bg};color:{fg};padding:5px 14px;border-radius:20px;
                          font-size:11px;font-weight:700;letter-spacing:1px;
                          text-transform:uppercase;\">{environment}</span></td>
        </tr></table></td></tr>
      <tr><td style=\"background:#f7f9fc;padding:16px 36px;border-bottom:1px solid #e4e9f0;\">
        <p style=\"margin:0;font-size:13.5px;color:#555;line-height:1.5;\">Hi Team,<br>
        Please find below the <strong style=\"color:#002e6e;\">{month_label} Executive Summary</strong>
        for the IKG Insights Release. This report is auto-generated by the STAAT Insights platform.
        </p></td></tr>
      <tr><td style=\"padding:28px 36px 20px 36px;\">{body}</td></tr>
      <tr><td style=\"padding:0 36px;\">
        <div style=\"height:2px;background:linear-gradient(to right,#002e6e,#7eb3e0,#eef1f6);\"></div>
      </td></tr>
      <tr><td style=\"padding:20px 36px 24px 36px;background:#f7f9fc;\">
        <table width=\"100%\" cellpadding=\"0\" cellspacing=\"0\"><tr>
          <td style=\"vertical-align:top;\">
            <div style=\"font-size:13px;font-weight:700;color:#002e6e;margin-bottom:3px;\">STAAT Insights Team</div>
            <div style=\"font-size:12px;color:#8a9bb5;line-height:1.6;\">Generated on {gen}<br>
              Environment: <span style=\"color:{bg};font-weight:600;\">{environment}</span></div></td>
          <td style=\"text-align:right;vertical-align:top;font-size:11px;color:#b0bec5;line-height:1.6;\">
            This is an automated report.<br>Please do not reply to this email.</td>
        </tr></table></td></tr>
    </table></td></tr>
</table></body></html>"""


print('All helpers loaded successfully.')

In [ ]:
# ── Cell 4: Choose a month ───────────────────────────────────────────────
month_options = get_month_options_from_db()
print('Available months (latest first):')
for i,o in enumerate(month_options, 1):
    print(f"  {i:>2}. {o['label']}  ({o['value']})")
print()
raw = input("Enter month (e.g. 'May 2026', 'May-2026', '2026-05'): ").strip()
try:
    SELECTED_MONTH = str(pd.Period(raw, 'M'))
except Exception:
    SELECTED_MONTH = datetime.strptime(raw.replace('-',' ').replace('/',' '),'%B %Y').strftime('%Y-%m')
SELECTED_LABEL = pd.Period(SELECTED_MONTH,'M').to_timestamp().strftime('%B %Y')
print(f'\nSelected: {SELECTED_LABEL}  [{SELECTED_MONTH}]')

In [ ]:
# ── Cell 5: Query GP and inspect ──────────────────────────────────────────
df_current, df_next_month, df_reactivated = get_exec_summary_data(SELECTED_MONTH)
NEXT_LABEL = (pd.Period(SELECTED_MONTH,'M')+1).to_timestamp().strftime('%B %Y')
print(f'Current month ({SELECTED_LABEL}) rows : {len(df_current)}')
print(f'Next month    ({NEXT_LABEL}) rows : {len(df_next_month) if df_next_month is not None else 0}')
print(f'Reactivated types                 : {0 if df_reactivated.empty else df_reactivated["rule_name"].nunique()}')
print('\n── Current month (first 10) ──'); display(df_current.head(10))
print('\n── Label breakdown ──')
display(df_current['labels'].dropna().str.split(',').explode().str.strip().value_counts())
print("\n── 'Top Feature' rows ──")
display(df_current[df_current['labels'].fillna('').apply(lambda v:_has_label(v,'Top Feature'))][['id_x','title','labels','rule_name']])
print("\n── 'New Insight' rows ──")
display(df_current[df_current['labels'].fillna('').apply(lambda v:_has_label(v,'New Insight'))][['id_x','title','labels','rule_name']])
print('\n── Reactivated insights ──')
if df_reactivated.empty: print('  (none found for this month)')
else:
    display(df_reactivated[['rule_name','id_x','title','issue_summary','iteration_end_date']])
    print('\nFormatted reactivated block:'); print(_build_reactivated_text(df_reactivated))

In [ ]:
# ── Cell 6: Generate the executive summary ───────────────────────────────
print(f'Calling {MODEL_NAME} \u2026 may take ~30 s.')
SUMMARY_TEXT = generate_summary(
    df_current,
    next_month_df  = df_next_month,
    reactivated_df = df_reactivated if not df_reactivated.empty else None,
)
print('\n' + '='*80)
print(f'Executive Summary \u2013 {SELECTED_LABEL}')
print('='*80 + '\n')
print(SUMMARY_TEXT)

In [ ]:
# ── Cell 7: Save to .txt (optional) ───────────────────────────────────────
content  = build_save_content(SUMMARY_TEXT, SELECTED_LABEL)
filename = f'executive_summary_{SELECTED_MONTH}.txt'
with open(filename, 'w', encoding='utf-8') as f: f.write(content)
print(f'Saved \u2192 {filename}')

In [ ]:
# ── Cell 8: Build & preview email HTML locally ────────────────────────────
#
# Locally: no email is sent (Airflow SMTP not available).
# The HTML is saved as a file so you can open it in a browser to
# verify the styling before deploying.
# In Airflow: generate_executive_summary() calls _send_exec_summary_email() automatically.

PREVIEW_ENV = 'Dev'   # change to 'UAT' or 'Prod' to preview that badge colour

html_content = build_email_html(SUMMARY_TEXT, SELECTED_LABEL, PREVIEW_ENV)
html_file    = f'exec_summary_email_preview_{SELECTED_MONTH}.html'

with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'HTML email preview saved \u2192 {html_file}')
print(f'Open it in a browser to review styling.\n')
print(f'Email subject (example): "Executive Summary \u2013 {SELECTED_LABEL} \u2013 {PREVIEW_ENV}"')
print(f'Recipients (local/Dev) : {LOCAL_EMAIL_RECIPIENTS}')
print(f'Recipients (Prod)      : Airflow Variable "{EMAIL_RECIPIENTS_VAR}"')